In [1]:
from test.helpers import batch_sde_solve, l2_dist, sde_solver_order

import jax
import jax.numpy as jnp
import jax.random as jrandom
import matplotlib.pyplot as plt
from diffrax import (
    ALIGN,
    ControlTerm,
    diffeqsolve,
    Euler,
    MultiTerm,
    ODETerm,
    PIDController,
    SaveAt,
    SEA,
    ShARK,
    VirtualBrownianTree,
)


jnp.set_printoptions(precision=4, suppress=True)


def show_conv_results(hs, errs):
    plt.plot(1 / hs, errs)
    plt.yscale("log")
    plt.xscale("log")
    plt.xticks(ticks=1 / hs, labels=hs)
    plt.ylabel("RMS error")
    plt.xlabel("step size")
    plt.show()
    slope, _ = jnp.polyfit(jnp.log(hs), jnp.log(errs), 1)
    print(f"Order of convergence: {slope:.4f}")


def plot_sol(sol, dim=1):
    for i in range(dim):
        plt.plot(sol.ts, sol.ys[:, i], label=f"x{i+1}")
        plt.plot(sol.ts, sol.ys[:, dim + i], label=f"v{i+1}")
    plt.legend()
    plt.show()


num_samples = 1000
keys = jrandom.split(jrandom.PRNGKey(2), num=num_samples)

## Harmonic Oscillator SDE

Given by 
\begin{align*}
    d \mathbf{x}_t &= \mathbf{v}_t \, dt \\
    d \mathbf{v}_t &= - \gamma \, \mathbf{v}_t \, dt - u \, \mathbf{x}_t \, dt + \sqrt{2 \gamma u} \, d W_t
\end{align*}
where $\mathbf{x}_t, \mathbf{v}_t, W_t \in \mathbb{R}^2$.

In [2]:
def drift(t, y, args):
    gamma, u, grad_f = args
    dim = int(y.shape[0] / 2)
    assert y.shape[0] == 2 * dim
    x, v = y[:dim], y[dim:]
    d_x = v
    d_v = -gamma * v - u * grad_f(x)
    d_y = jnp.array([d_x, d_v], dtype=y.dtype).flatten()
    return d_y


def diffusion(t, y, args):
    gamma, u, _ = args
    dim = int(y.shape[0] / 2)
    assert y.shape[0] == 2 * dim
    d_v = jnp.sqrt(2 * gamma * u) * jnp.ones((dim,), dtype=y.dtype)
    d_y = jnp.concatenate((jnp.zeros((dim, dim), dtype=y.dtype), jnp.diag(d_v)), axis=0)
    return d_y


def get_terms(bm: VirtualBrownianTree):
    return MultiTerm(ODETerm(drift), ControlTerm(diffusion, bm))


gamma = jnp.array([2, 0.5], dtype=jnp.float32)
u = jnp.array([0.5, 2], dtype=jnp.float32)
args = (gamma, u, lambda x: 2 * x)
y0_hosc = jnp.array([0, 0, 0, 0], dtype=jnp.float32)
t0, t1 = 0.3, 15
w_dim_hosc = 2
# Langevin = (drift, diffusion, args, y0, t0, t1, dim)
harmonic_osc = (drift, diffusion, args, y0_hosc, t0, t1, w_dim_hosc)

In [3]:
hs, ALIGN_errs, order = sde_solver_order(
    keys, harmonic_osc, ALIGN(0.1), ALIGN(0.1), 0.005, hs_num=5
)

RuntimeError: jaxlib/gpu/solver_kernels.cc:45: operation gpusolverDnCreate(&handle) failed: cuSolver internal error

In [ ]:
show_conv_results(hs, ALIGN_errs)

In [ ]:
print(
    f"Each time the step size halves, the "
    f"error falls by a factor of: {ALIGN_errs[1:]/ALIGN_errs[:-1]}"
)

Now compare the RMS error of different methods when using $dt = 0.1$, against a "correct solution" created vie fine-grained Euler.

In [ ]:
correct_sol = batch_sde_solve(keys, harmonic_osc, dt0=0.005, solver=Euler())
euler_sol = batch_sde_solve(keys, harmonic_osc, dt0=0.1, solver=Euler())
sea_sol = batch_sde_solve(keys, harmonic_osc, dt0=0.1, solver=SEA())
shark_sol = batch_sde_solve(keys, harmonic_osc, dt0=0.1, solver=ShARK())
align_sol = batch_sde_solve(keys, harmonic_osc, dt0=0.1, solver=ALIGN(0.1))
align_pid_sol = batch_sde_solve(
    keys,
    harmonic_osc,
    dt0=0.1,
    solver=ALIGN(0.1),
    stepsize_controller=PIDController(rtol=0.001, atol=0.0003),
)

In [ ]:
print(f"Euler       {l2_dist(correct_sol,euler_sol):.4f}")
print(f"SEA         {l2_dist(correct_sol,sea_sol):.4f}")
print(f"ShARK       {l2_dist(correct_sol,shark_sol):.4f}")
print(f"ALIGN       {l2_dist(correct_sol,align_sol):.4f}")
print(f"ALIGN_PID   {l2_dist(correct_sol,align_pid_sol):.4f}")

In [ ]:
bm = VirtualBrownianTree(
    t0, t1, tol=2**-9, shape=(w_dim_hosc,), key=jrandom.PRNGKey(2), compute_stla=True
)
terms = MultiTerm(ODETerm(drift), ControlTerm(diffusion, bm))
solALIGN = diffeqsolve(
    terms,
    ALIGN(0.1),
    t0,
    t1,
    dt0=0.1,
    y0=y0_hosc,
    args=args,
    saveat=SaveAt(ts=jnp.linspace(t0, t1, 1000)),
)
plot_sol(solALIGN, dim=2)

## Timing for Euler, ALIGN, and ShARK

In [ ]:
# EULER 1000 samples
jax.block_until_ready(batch_sde_solve(keys, harmonic_osc, dt0=0.1, solver=Euler()))
%timeit jax.block_until_ready(batch_sde_solve(keys, harmonic_osc, dt0=0.1, solver=Euler()))

In [ ]:
# ShARK 1000 samples
jax.block_until_ready(batch_sde_solve(keys, harmonic_osc, dt0=0.1, solver=ShARK()))
%timeit jax.block_until_ready(batch_sde_solve(keys, harmonic_osc, dt0=0.1, solver=ShARK()))

In [ ]:
# ALIGN 1000 samples
jax.block_until_ready(batch_sde_solve(keys, harmonic_osc, dt0=0.1, solver=ALIGN(0.1)))
%timeit jax.block_until_ready(batch_sde_solve(keys, harmonic_osc, dt0=0.1, solver=ALIGN(0.1)))

In [ ]:
# ALIGN 10x more steps, 1000 samples
jax.block_until_ready(batch_sde_solve(keys, harmonic_osc, dt0=0.01, solver=ALIGN(0.1)))
%timeit jax.block_until_ready(batch_sde_solve(keys, harmonic_osc, dt0=0.01, solver=ALIGN(0.1)))

In [ ]:
# ALIGN 10 samples
keys_short = jrandom.split(jrandom.PRNGKey(2), num=10)
jax.block_until_ready(
    batch_sde_solve(keys_short, harmonic_osc, dt0=0.1, solver=ALIGN(0.1))
)
%timeit jax.block_until_ready(batch_sde_solve(keys_short, harmonic_osc, dt0=0.1, solver=ALIGN(0.1)))

In [ ]:
# ALIGN 100,000 samples
keys_long = jrandom.split(jrandom.PRNGKey(2), num=100000)
jax.block_until_ready(
    batch_sde_solve(keys_long, harmonic_osc, dt0=0.1, solver=ALIGN(0.1))
)
%timeit jax.block_until_ready(batch_sde_solve(keys_long, harmonic_osc, dt0=0.1, solver=ALIGN(0.1)))

## Bistable Quartic potential

Given by 
\begin{align*}
    d x_t &= v_t \, dt \\
    d v_t &= - \gamma \, v_t \, dt - u \, \nabla f( x_t ) \, dt + \sqrt{2 \gamma u} \, d W_t
\end{align*}
where $x_t, v_t, W_t \in \mathbb{R}$ and $f(x) = (x-1)^2 (x+1)^2$.

In [ ]:
grad_f = lambda x: 4 * x * (jnp.square(x) - 1)

args_bqp = (jnp.float32(0.8), jnp.float32(0.2), grad_f)
y0_bqp = jnp.array([0, 0], dtype=jnp.float32)
t0, t1 = 0.3, 15
w_dim_bqp = 1
bqp = (drift, diffusion, args_bqp, y0_bqp, t0, t1, w_dim_bqp)

In [ ]:
bm = VirtualBrownianTree(
    t0, t1, tol=2**-9, shape=(), key=jrandom.PRNGKey(4), compute_stla=True
)
terms = MultiTerm(ODETerm(drift), ControlTerm(diffusion, bm))
solALIGN = diffeqsolve(
    terms,
    ALIGN(0.1),
    t0,
    t1,
    dt0=0.05,
    y0=y0_bqp,
    args=args_bqp,
    saveat=SaveAt(ts=jnp.linspace(t0, t1, 1000)),
)
plot_sol(solALIGN, dim=1)

In [ ]:
hs, ALIGN_errs_bqp, _ = sde_solver_order(keys, bqp, ALIGN(0.1), ALIGN(0.1), 0.005)
show_conv_results(hs, ALIGN_errs_bqp)

In [ ]:
print(hs)
print(ALIGN_errs_bqp)

In [ ]:
correct_sol_bqp = batch_sde_solve(keys, bqp, dt0=0.005, solver=Euler())
euler_sol_bqp = batch_sde_solve(keys, bqp, dt0=0.1, solver=Euler())
shark_sol_bqp = batch_sde_solve(keys, bqp, dt0=0.1, solver=ShARK())
align_sol_bqp = batch_sde_solve(keys, bqp, dt0=0.1, solver=ALIGN(0.1))

print(f"Euler       {l2_dist(correct_sol_bqp, euler_sol_bqp):.4f}")
print(f"ShARK       {l2_dist(correct_sol_bqp, shark_sol_bqp):.4f}")
print(f"ALIGN       {l2_dist(correct_sol_bqp, align_sol_bqp):.4f}")

## Adaptive stepping with a PID controller

In [ ]:
bm = VirtualBrownianTree(
    t0, t1, tol=2**-9, shape=(w_dim_hosc,), key=jrandom.PRNGKey(2), compute_stla=True
)
terms = MultiTerm(ODETerm(drift), ControlTerm(diffusion, bm))
stepsize_controller = PIDController(rtol=0.003, atol=0.001)
solALIGN_PID = diffeqsolve(
    terms,
    ALIGN(0.1),
    t0,
    t1,
    dt0=1.0,
    y0=y0_hosc,
    args=args,
    saveat=SaveAt(ts=jnp.linspace(t0, t1, 1000)),
    stepsize_controller=stepsize_controller,
)
plot_sol(solALIGN_PID, dim=2)